In [1]:
import io
import requests
import pandas as pd
import yfinance as yf
import numpy as np
import gdown

Question 1

In [2]:
#Load Data
url = "https://www.iposcoop.com/ipos-recently-filed/"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers)

# Use io.StringIO to avoid lxml/Windows OSError
df = pd.read_html(io.StringIO(resp.text))[0]

# Filter for 'Withdrawn'
df = df[df["Expected To Trade"] == "Withdrawn"].copy()
df.head()

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
2,2026-09-25,Holtec Nuclear (Withdrawn),HNUC,J.P.Morgan/Guggenheim Securities/Goldman Sachs...,50.00,$15.00,$18.00,$825.00,Withdrawn,S/O
25,2026-09-18,Rise Smart Group (Withdrawn),RSHL,American Trust Investment Services/Prime Numbe...,1.80,$4.00,$4.00,$7.20,Withdrawn,S/O
28,2026-09-17,C2 Capital Group (Withdrawn),CCLV,Dominari Securities,3.75,$4.00,$5.00,$16.88,Withdrawn,S/O
40,2026-09-15,OTSAW Ltd. (Withdrawn),OTSA,Aegis Capital,4.40,$4.50,$5.50,$22.00,Withdrawn,S/O
46,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,$100.00,Withdrawn,S/O


In [3]:
# Company Classification (Order matters!)
def classify_company(name):
    if not isinstance(name, str):
        return "Other"
    if "Technologies" in name:
        return "Technologies"
    elif any(x in name for x in ["Acquisition Corp", "Acquisition Corporation", "Corp"]):
        return "Acquisition Corp"
    elif any(x in name for x in ["Inc", "Incorporated"]):
        return "Inc."
    elif "Group" in name:
        return "Group"
    elif any(x in name for x in ["Ltd", "Limited"]):
        return "Limited"
    elif any(x in name for x in ["Holdings", "Holding"]):
        return "Holdings"
    else:
        return "Other"


df["Company Type"] = df["Company"].apply(classify_company)

In [4]:
# Parse Prices & Clean Numerics
def parse_avg_price(row):
    try:
        low = float(str(row["Price Low"]).replace("$", "").strip())
        high = float(str(row["Price High"]).replace("$", "").strip())
        return (low + high) / 2
    except:
        return None


df["Avg_price"] = df.apply(parse_avg_price, axis=1)

# Clean numeric columns
for col in ["Shares (millions)", "Est $ Vol (millions)"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [5]:
# Shares_offered_value Calculation
df["Shares_offered_value"] = df["Shares (millions)"] * df["Avg_price"]
df["Shares_offered_value"] = df["Shares_offered_value"].fillna(
    df["Est $ Vol (millions)"]
)

# Aggregation
result = (
    df.groupby("Company Type")["Shares_offered_value"]
    .sum()
    .sort_values(ascending=False)
)
print("Total withdrawn IPO value by Company Type ($ Millions):")
print(result)

Total withdrawn IPO value by Company Type ($ Millions):
Company Type
Other               1115.4450
Acquisition Corp     499.9850
Inc.                 351.0000
Holdings             311.6575
Limited              219.2500
Technologies         184.9000
Group                 56.5750
Name: Shares_offered_value, dtype: float64


Question 2

In [6]:
# Download 2025 Pricings
url = "https://www.iposcoop.com/2025-pricings/"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers)
df_ipos = pd.read_html(io.StringIO(resp.text))[0]

df_ipos.head()

,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
0,Vine Hill Capital Investment Corp. II,VHCPU,Blank Check,12/18/2025,20.0,$10.00,$0.00,$0.00,0.00%,S/O
1,Andersen Group,ANDG,Consumer Services,12/17/2025,11.0,$16.00,$23.50,$23.79,48.69%,S/O
2,Medline Inc.,MDLN,Health Care,12/17/2025,216.0,$29.00,$41.00,$43.83,51.14%,S/O
3,Wealthfront Corp.,WLTH,Financials,12/12/2025,34.6,$14.00,$14.19,$8.49,-39.36%,S/O
4,Lumexa Imaging Holdings,LMRI,Health Care,12/11/2025,25.0,$18.50,$18.52,$14.80,-20.00%,S/O


In [7]:
# Filtering
df_ipos["Offer Date"] = pd.to_datetime(df_ipos["Offer Date"], errors="coerce")
filtered_ipos = df_ipos[
    (df_ipos["Offer Date"] < "2025-09-01") & (df_ipos["Return"] != "0.00%")
].copy()

tickers = filtered_ipos["Symbol"].dropna().tolist()

len(tickers)

146

In [8]:
# Download stock data via yfinance
#stocks_data = yf.download(tickers, start="2025-01-01", end="2026-09-12")

# Flatten MultiIndex columns if present (recent yfinance versions)
#if isinstance(stocks_data.columns, pd.MultiIndex):
#    close_df = stocks_data["Close"]
#else:
#    close_df = stocks_data["Close"]

# Clean delisted/empty columns
#close_df = close_df.dropna(how="all", axis=1)

#close_df.head()


stocks_data = yf.download(
    tickers, start="2025-01-01", end="2026-09-12", progress=False
)

# Handle MultiIndex and drop missing tickers
close_df = stocks_data["Close"]
if isinstance(close_df.columns, pd.MultiIndex):
    close_df = close_df.droplevel(0, axis=1)

close_df = close_df.dropna(how="all", axis=1)
close_df.head()

$SKBL: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EPWK"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: PTNM"}}}
$MTSR: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found
$PTNM: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found
$EPWK: possibly delisted; no timezone found
$AGH: possibly delisted; no timezone found
$MJID: possibly delisted; no timezone found
$WGRX: possibly delisted; no timezone found
$TBH: possibly delisted; no timezone found
$DLXY: possibly delisted; no price data found  (1d 2025-01-01 -> 2026-09-12)
$MCTR: possibly delisted; no timezone found
$EMPG: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found

15 Failed downloads:
['SKBL', 'MTSR', 'CEPT', 'PTNM', 'A

Ticker,AAPG,AARD,ADVB,AII,AIRO,ALM,AMBQ,ANPA,ANTA,APUS,...,VOYG,WETO,WFF,WTF,WXM,WYFI,XHLD,YMAT,YMT,ZYBT
Date,,,,,,,,,,,,,,,,,,,,,
2025-01-02,NaN,NaN,NaN,NaN,NaN,1.55250,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-01-03,NaN,NaN,NaN,NaN,NaN,1.62225,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-01-06,NaN,NaN,NaN,NaN,NaN,1.60200,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-01-07,NaN,NaN,NaN,NaN,NaN,1.59525,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.86
2025-01-08,NaN,NaN,NaN,NaN,NaN,1.59525,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.50


In [9]:
# Compute Growth, Volatility & Sharpe
growth_252d = close_df / close_df.shift(252)
volatility = close_df.rolling(30).std() * np.sqrt(252)
sharpe = (growth_252d - 0.05) / volatility

In [10]:
# Filter for target date '2026-09-11'
target_date = "2026-09-11"
if target_date in sharpe.index:
    sharpe_target = sharpe.loc[target_date].dropna()
else:
    sharpe_target = sharpe.iloc[-1].dropna()

print("--- Question 2: Sharpe Ratio Summary ---")
print(sharpe_target.describe())
print(f"\nMedian Sharpe Ratio on {target_date}: {sharpe_target.median():.4f}")

--- Question 2: Sharpe Ratio Summary ---
count    129.000000
mean            inf
std             NaN
min       -0.040147
25%        0.011942
50%        0.048009
75%        0.134248
max             inf
Name: 2026-09-11 00:00:00, dtype: float64

Median Sharpe Ratio on 2026-09-11: 0.0480


C:\Users\saulp\miniconda3\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


Question 3

In [11]:
# Reusing close_df from Question 2
holding_medians = {}

for m in range(1, 13):
    days = m * 21  # 1 month = 21 trading days
    # Growth relative to entry price (min_date closing price for each stock)
    # Find min_date (first valid trading day) for each ticker
    first_prices = close_df.bfill().iloc[0]

    # Shift future price by 'days' trading steps
    future_prices = close_df.shift(-days)
    growth_col = (future_prices / first_prices).iloc[0]

    # Calculate median across active tickers
    holding_medians[f"{m}_m"] = growth_col.median()

df_holding = pd.Series(holding_medians)
print("Median growth by holding period (in months):")
print(df_holding)

best_month = df_holding.idxmax()
print(
    f"\nOptimal holding period: {best_month} with median growth of {df_holding.max():.4f}"
)

Median growth by holding period (in months):
1_m     1.000498
2_m     1.000000
3_m     0.903743
4_m     0.828742
5_m     0.863636
6_m     0.863440
7_m     0.843842
8_m     0.864705
9_m     0.791709
10_m    0.642787
11_m    0.659595
12_m    0.627825
dtype: float64

Optimal holding period: 1_m with median growth of 1.0005


Question 4

In [12]:
# Download file
file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(
    f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False
)
df = pd.read_parquet("data.parquet", engine="pyarrow")

# Filter for RSI < 30 and Date range (2000-01-01 to 2025-06-01)
df["Date"] = pd.to_datetime(df["Date"])
selected_df = df[
    (df["Date"] >= "2000-01-01")
    & (df["Date"] <= "2025-06-01")
    & (df["rsi"] < 30)
]

# Net Income calculation
net_income = 1000 * (selected_df["growth_future_30d"] - 1).sum()
net_income_thousands = net_income / 1000

print(f"Total Number of Signals Triggered: {len(selected_df)}")
print(f"Net Income: ${net_income:,.2f}")
print(f"Net Income (in $ thousands): ${net_income_thousands:,.2f}k")

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=3406a80e-6726-478f-9708-06ce8179c265
To: C:\Users\saulp\Documents\Data Talks Club\Stock Market Analytics\Stock Markets Analytics - 2026\data.parquet
100%|███████████████████████████████████████████████████████████████████████████████| 130M/130M [00:13<00:00, 9.66MB/s]


Total Number of Signals Triggered: 5206
Net Income: $65,805.59
Net Income (in $ thousands): $65.81k
